In [ ]:
"""
CIFAR-10 통합 실습 스크립트 — ver0
=====================================================================
Sources integrated:
  - 15차시: CNN 기초 모델링
  - 17차시: Batch Normalization / Dropout / Pooling / Data Augmentation
  - 18차시: Transfer Learning (ResNet50, frozen backbone)

Design principles:
  - 단일 데이터 경로(CustomDataset + torchvision.transforms) 사용
  - CrossEntropyLoss는 logits 입력만 받음 (Softmax/Sigmoid 제거)
  - 모듈화된 train/eval 엔진을 두 모델이 공유
  - 시드 고정 + cudnn 결정성 설정으로 재현성 확보
=====================================================================
"""



---


#실습 목적
*   접근법 A (**Custom CNN**): 내 데이터에 맞춰 작고 가벼운 모델을 처음부터 직접 짠다.
    *   한계: 데이터와 모델이 작아서 71%에서 성능이 멈춤

<br>

*   접근법 B (Transfer Learning, ResNet50): 남이 만들어둔 거대하고 똑똑한 모델을 가져와서, 내 데이터를 그 모델의 입맛(224x224, ImageNet 정규화)에 억지로 맞춰서라도 학습시킨다.
    *   결과: 겨우 5 Epoch 만에 84.8%라는 성능 달성 가능

    <br>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from __future__ import annotations

In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets import CIFAR10

In [ ]:
# =====================================================================
# 1. Reproducibility
# =====================================================================
def set_seed(seed: int = 2023) -> None:
    """Seed Python/Numpy/Torch (CPU & CUDA) and force cuDNN determinism."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# =====================================================================
# 2. Configuration
# =====================================================================
@dataclass
class Config:
    seed: int = 2023
    data_root: str = "./data"
    val_ratio: float = 0.2
    num_classes: int = 10

    # Custom CNN
    custom_input_size: int = 32
    epochs_custom: int = 30
    lr_custom: float = 1e-3
    dropout_p: float = 0.3

    # Transfer learning
    pretrained_input_size: int = 224
    epochs_finetune: int = 5  # 소량 epoch로도 frozen-fc는 빠르게 수렴
    lr_finetune: float = 1e-3

    # Loader
    batch_size: int = 128
    num_workers: int = 2

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# =====================================================================
# 3. Data loading
# =====================================================================
def load_cifar10_splits(cfg: Config) -> Tuple[
    Tuple[np.ndarray, np.ndarray],
    Tuple[np.ndarray, np.ndarray],
    Tuple[np.ndarray, np.ndarray],
]:
    """
    torchvision.datasets.CIFAR10로 raw numpy(HWC, uint8)를 확보 후
    stratified train/val split.

    Returns:
        (x_train, y_train), (x_val, y_val), (x_test, y_test)
        - 이미지: uint8 ndarray (N, 32, 32, 3)
        - 라벨:   int64 ndarray (N,)
    """
    train_full = CIFAR10(root=cfg.data_root, train=True, download=True)
    test_set = CIFAR10(root=cfg.data_root, train=False, download=True)

    x_full = train_full.data  # (50000, 32, 32, 3) uint8 HWC
    y_full = np.array(train_full.targets, dtype=np.int64)
    x_test = test_set.data
    y_test = np.array(test_set.targets, dtype=np.int64)

    x_train, x_val, y_train, y_val = train_test_split(
        x_full,
        y_full,
        test_size=cfg.val_ratio,
        random_state=cfg.seed,
        stratify=y_full,
    )
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

In [ ]:
# =====================================================================
# 4. Custom Dataset
# =====================================================================
class CIFAR10Dataset(Dataset):
    """
    numpy (HWC uint8) 이미지 배열을 PIL로 변환 후 transform 적용.
    transform 파이프라인의 ToTensor()가 CHW float[0,1] 텐서를 생성하므로
    모델 forward 내부에서는 별도 permute가 필요 없음.
    """

    def __init__(
        self,
        images: np.ndarray,
        labels: np.ndarray,
        transform: Callable | None = None,
    ) -> None:
        assert len(images) == len(labels), "images/labels length mismatch"
        self.images = images
        self.labels = labels.astype(np.int64)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img = Image.fromarray(self.images[idx])
        label = int(self.labels[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, label

In [ ]:
# =====================================================================
# 5. Transforms
# =====================================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
def build_transforms(
    input_size: int,
    augment: bool,
    normalize_imagenet: bool = False,
) -> transforms.Compose:
    """
    공통 transform 빌더.
      - augment=True: train용 (Flip + 소폭 회전 + ColorJitter)
      - normalize_imagenet=True: pretrained 경로용 ImageNet stats 적용
    원본의 RandomVerticalFlip + 90° Rotation은 CIFAR-10(자연 이미지)에
    과도하므로 제거.
    """
    ops: List[Callable] = [transforms.Resize((input_size, input_size))]
    if augment:
        ops += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    ops.append(transforms.ToTensor())  # → CHW float[0,1]
    if normalize_imagenet:
        ops.append(transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD))
    return transforms.Compose(ops)

In [ ]:
# =====================================================================
# 6. DataLoader builder
# =====================================================================
def build_loaders(
    splits: Tuple[Tuple[np.ndarray, np.ndarray], ...],
    cfg: Config,
    input_size: int,
    normalize_imagenet: bool,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    (x_tr, y_tr), (x_va, y_va), (x_te, y_te) = splits

    train_tf = build_transforms(input_size, augment=True, normalize_imagenet=normalize_imagenet)
    eval_tf = build_transforms(input_size, augment=False, normalize_imagenet=normalize_imagenet)

    train_ds = CIFAR10Dataset(x_tr, y_tr, transform=train_tf)
    val_ds = CIFAR10Dataset(x_va, y_va, transform=eval_tf)
    test_ds = CIFAR10Dataset(x_te, y_te, transform=eval_tf)

    pin = cfg.device == "cuda"
    common = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=pin)

    train_loader = DataLoader(train_ds, shuffle=True, drop_last=False, **common)
    val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **common)
    test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **common)
    return train_loader, val_loader, test_loader

In [ ]:
# =====================================================================
# 7. Models
# =====================================================================
class CustomCNN(nn.Module):
    """
    Conv → BN → ReLU → MaxPool 구조 2블록 + Dropout + Linear.
    입력: (B, 3, 32, 32). 출력: (B, num_classes) logits.

    Shape trace (k=3, s=1, padding=0; pool k=2, s=2):
        32x32x3
          → conv1 → 30x30x32 → pool → 15x15x32
          → conv2 → 13x13x64 → pool → 6x6x64
          → flatten → 2304
          → dropout → linear → num_classes
    """

    def __init__(self, num_classes: int = 10, dropout_p: float = 0.3) -> None:
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.dropout = nn.Dropout(p=dropout_p)
        self.classifier = nn.Linear(in_features=6 * 6 * 64, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x is already (B, C, H, W) thanks to ToTensor() in the transform.
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(start_dim=1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x  # raw logits — CE applies log-softmax internally

In [ ]:
def build_resnet50_transfer(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    ImageNet-pretrained ResNet50을 로드하고 fc head만 num_classes로 교체.
    freeze_backbone=True인 경우 fc 외 모든 파라미터는 requires_grad=False.
    """
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Linear(in_features, num_classes)

    if freeze_backbone:
        for name, param in model.named_parameters():
            param.requires_grad = name.startswith("fc.")
    return model

In [ ]:
def trainable_parameters(model: nn.Module) -> List[nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]

In [ ]:
# =====================================================================
# 8. Train / Eval engine
# =====================================================================
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: str,
) -> Tuple[float, float]:
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=-1)
        total_correct += (preds == y).sum().item()
        total_samples += x.size(0)

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> Tuple[float, float]:
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=-1)
        total_correct += (preds == y).sum().item()
        total_samples += x.size(0)

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: str,
    epochs: int,
    tag: str = "",
) -> Dict[str, List[float]]:
    history: Dict[str, List[float]] = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(
            f"[{tag}] {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )
    return history

In [ ]:
# =====================================================================
# 9. Optional: history plot
# =====================================================================
def plot_history(histories: Dict[str, Dict[str, List[float]]], save_path: str | None = None) -> None:
    """English titles/labels to avoid Korean font rendering issues on matplotlib."""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for tag, h in histories.items():
        axes[0].plot(h["train_loss"], label=f"{tag} train")
        axes[0].plot(h["val_loss"], label=f"{tag} val", linestyle="--")
        axes[1].plot(h["train_acc"], label=f"{tag} train")
        axes[1].plot(h["val_acc"], label=f"{tag} val", linestyle="--")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# 10. Main
# =====================================================================
def main() -> Dict[str, Dict]:
    cfg = Config()
    set_seed(cfg.seed)
    print(f"Device: {cfg.device}")

    splits = load_cifar10_splits(cfg)
    criterion = nn.CrossEntropyLoss()

    # -----------------------------------------------------------------
    # (A) Custom CNN @ 32x32 — BN + Pool + Dropout + Augmentation
    # -----------------------------------------------------------------
    print("\n=== A. Custom CNN (BN + Pool + Dropout + Aug) ===")
    train_loader_a, val_loader_a, test_loader_a = build_loaders(
        splits, cfg, input_size=cfg.custom_input_size, normalize_imagenet=False
    )
    model_a = CustomCNN(num_classes=cfg.num_classes, dropout_p=cfg.dropout_p).to(cfg.device)
    optim_a = optim.Adam(model_a.parameters(), lr=cfg.lr_custom)
    history_a = fit(
        model_a, train_loader_a, val_loader_a, criterion, optim_a,
        cfg.device, cfg.epochs_custom, tag="CustomCNN",
    )
    test_loss_a, test_acc_a = evaluate(model_a, test_loader_a, criterion, cfg.device)
    print(f"[CustomCNN] TEST loss {test_loss_a:.4f} acc {test_acc_a:.4f}")

    # -----------------------------------------------------------------
    # (B) Transfer Learning @ 224x224 — ResNet50, frozen backbone
    # -----------------------------------------------------------------
    print("\n=== B. Transfer Learning (ResNet50, frozen backbone) ===")
    train_loader_b, val_loader_b, test_loader_b = build_loaders(
        splits, cfg, input_size=cfg.pretrained_input_size, normalize_imagenet=True
    )
    model_b = build_resnet50_transfer(
        num_classes=cfg.num_classes, freeze_backbone=True
    ).to(cfg.device)
    optim_b = optim.Adam(trainable_parameters(model_b), lr=cfg.lr_finetune)
    history_b = fit(
        model_b, train_loader_b, val_loader_b, criterion, optim_b,
        cfg.device, cfg.epochs_finetune, tag="ResNet50",
    )
    test_loss_b, test_acc_b = evaluate(model_b, test_loader_b, criterion, cfg.device)
    print(f"[ResNet50] TEST loss {test_loss_b:.4f} acc {test_acc_b:.4f}")

    results = {
        "custom_cnn": {"history": history_a, "test_loss": test_loss_a, "test_acc": test_acc_a},
        "resnet50": {"history": history_b, "test_loss": test_loss_b, "test_acc": test_acc_b},
    }

    # 학습 곡선 시각화 (선택)
    # plot_history({"CustomCNN": history_a, "ResNet50": history_b}, save_path="curves.png")

    return results

In [ ]:
if __name__ == "__main__":
    main()

Device: cuda


100%|██████████| 170M/170M [00:03<00:00, 43.8MB/s]



=== A. Custom CNN (BN + Pool + Dropout + Aug) ===
[CustomCNN] 01/30 | train loss 1.4865 acc 0.4668 | val loss 1.2248 acc 0.5778
[CustomCNN] 02/30 | train loss 1.2177 acc 0.5671 | val loss 1.1711 acc 0.5929
[CustomCNN] 03/30 | train loss 1.1285 acc 0.6025 | val loss 1.3191 acc 0.5525
[CustomCNN] 04/30 | train loss 1.0687 acc 0.6246 | val loss 1.2797 acc 0.5645
[CustomCNN] 05/30 | train loss 1.0357 acc 0.6391 | val loss 0.9283 acc 0.6765
[CustomCNN] 06/30 | train loss 0.9982 acc 0.6535 | val loss 0.9551 acc 0.6727
[CustomCNN] 07/30 | train loss 0.9766 acc 0.6592 | val loss 0.9837 acc 0.6615
[CustomCNN] 08/30 | train loss 0.9655 acc 0.6641 | val loss 0.8859 acc 0.6983
[CustomCNN] 09/30 | train loss 0.9376 acc 0.6746 | val loss 0.8712 acc 0.7017
[CustomCNN] 10/30 | train loss 0.9216 acc 0.6802 | val loss 1.0532 acc 0.6366
[CustomCNN] 11/30 | train loss 0.9128 acc 0.6816 | val loss 0.8414 acc 0.7129
[CustomCNN] 12/30 | train loss 0.8975 acc 0.6879 | val loss 0.9887 acc 0.6632
[CustomCNN] 1

100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]


[ResNet50] 01/5 | train loss 1.0514 acc 0.6923 | val loss 0.6280 acc 0.8082
[ResNet50] 02/5 | train loss 0.7085 acc 0.7703 | val loss 0.5270 acc 0.8339
[ResNet50] 03/5 | train loss 0.6408 acc 0.7897 | val loss 0.4972 acc 0.8389
[ResNet50] 04/5 | train loss 0.6057 acc 0.7991 | val loss 0.4782 acc 0.8411
[ResNet50] 05/5 | train loss 0.5821 acc 0.8056 | val loss 0.4538 acc 0.8491
[ResNet50] TEST loss 0.4527 acc 0.8480


In [ ]:
from IPython.display import HTML

# 1. html 파일이 있는 구글 드라이브 경로를 정확히 입력하세요.
html_path = '/content/drive/MyDrive/Colab Notebooks/PyTorch 실습 /cifar10_training_curves_comparison.html'

# 2. 파일을 파이썬으로 열어서 내용을 읽습니다.
with open(html_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 3. 셀 아래에 출력합니다.
display(HTML(html_content))

#1. 수치 요약

```
# CustomCNN (30 ep) vs ResNet50 frozen (5 ep)
```
*   Final train acc: 0.7281 vs 0.8056 (+0.0775)
*   FInal val acc: 0.1789 vs 0.8491 (+0.1302)
*  **Test acc: 0.7155 vs 0.8480 (+0.1325)**
*   Final train loss: 0.7861 vs 0.5821 (-0.2040)
*   Final val loss: 0.8507 vs 0.4538 (-0.3969)
*   **Test loss: 0.8583 vs 0.4527 (-0.4056)**


<br>

*   **13.25%p의 격차 의미**
*   : ResNet50 frozen-fc가 **6배 적은 epoch (5 vs 30) 으로 CustomCNN보다 test에서 +13.25%p 우수했음**
    *   **ImageNet pre-training으로 학습된 일반 visual feature가 CIFAR-10에 강하게 transfer됨을 시사**
    *   입력 해상도가 다름(32×32 vs 224×224)에 유의: 두 모델은 동일 입력 분포에서 비교된 것이 아님. 따라서 같은 해상도에서 같은 정규화를 거친 데이터를 가지고, 아키텍처의 성능만 비교하는 **완벽한 비교 실험은 아닌 것**임.
        *   CIFAR-10 데이터셋의 해상도: 32×32
        *   ImageNet 데이터의 기본 규격: 224×224
        *   ResNet50을 통한 전이 학습을 실습하기 위해, 32×32 원본이미지를 강제로 224×224로 resize하여 입력한 것임을 유의할 것.




---


#2. CustomCNN 진단
*   관찰 1 — **학습 후반 plateau** Train acc:
    *   21 epoch (0.7157) → 30 epoch (0.7281), 9 epoch 동안 +1.24%p에 그침. 학습률 상수(Adam 1e-3)와 모델 capacity의 한계가 동시에 작용하는 것으로 추정.
    *   CIFAR-10 데이터셋의 원본 해상도는 32×32이다. 내가 직접 만든 CustomCNN (2-block 아키텍처)는 이 원본 사이즈를 그대로 받아들여 맨 땅에서부터, from-scratch 학습하도록 설계함.
    *   그렇다보니, 25 epoch 부근에서 Val acc가 0.75 부근을 찍고 더 이상 오르지 못하며 맴도는 plateau 추세를 보임.
    *    컨볼루션 층이 2개 (2-block) 밖에 존재하지 않는 얕은 신경망이다 보니, 아무리 Data Augmentation을 한다고 해서, 고양이/강아지/비행기 등의 복잡하고 깊은 특징 (features)를 다 담아낼 **Capacity가 부족**한 것임.
  
<br>

*   관찰 2 — **Val 변동성이 매우 큼**
    *   Epoch 13: val_acc=0.6306 (저점)
    *   Epoch 25: val_acc=0.7514 (정점)
    *   Epoch 30: val_acc=0.7189
    *   Train acc는 단조증가지만, Val acc는 ±0.05 범위에서 진동.
    *   **Best-model checkpoint를 저장하지 않았으므로** 정점(0.7514) 대비 3.25%p 손실된 가중치로 test를 평가한 결과.

<br>

*   관찰 3 — Train-Val gap이 거의 없음
    *   Final gap +0.92%p는 overfitting 신호로 보기 어려움. 오히려 under-fitting 또는 capacity-limited 구간에 있음.

<br>

*   관찰 4 — Test-Val 일관성
    *   Test acc (0.7155)와 Final val acc (0.7189)의 차이가 0.34%p로 매우 작음 → 데이터 split이 잘 분포되어 있고, val이 test의 신뢰할 만한 proxy임을 시사.



---


#3. ResNet50 진단
*   관찰 5 — Val > Train 현상
    *   **모든 epoch에서 val_acc > train_acc**. 비정상이 아니라 다음 요인들의 조합:
        *   Train transform에 RandomHorizontalFlip + RandomRotation + ColorJitter 포함 → train이 본질적으로 더 어려움
        *   Eval 시 model.eval() 모드에서 dropout off (단, frozen backbone이고 fc만 학습이므로 dropout은 없음. BatchNorm running stats만 영향)
        *   매 epoch 끝의 train acc는 학습 중 평균이지만, val은 학습 직후 단일 평가

<br>

*   관찰 6 — **Val loss 단조적 개선**, 아직 plateau 아님
    *   Val loss: 0.6280 → 0.5270 → 0.4972 → 0.4782 → 0.4538. 5 epoch이 끝나는 시점에도 계속 감소 중. 더 학습할 여지가 큼.

<br>

*   관찰 7 — **Test-Val 일관성**
    *   Test acc (0.8480) vs Final val acc (0.8491) 차이 0.11%p. 매우 안정.



---


#4. 식별된 한계점 (현재 ver0 스크립트 기준)
*   **Best-model checkpoint 부재**: 마지막 epoch 가중치로 test 평가. CustomCNN의 경우 epoch 25 (val_acc 0.7514) 가 정점이었으나 epoch 30 (val_acc 0.7189) 가중치 사용.
*   **LR scheduler 부재**: 후반 plateau에서 LR decay가 없어 fine-tuning 단계의 개선 여지 손실.
*   단일 seed 실행: 통계적 변동성 정량화 불가. CustomCNN의 val 변동성을 보면 seed별 ±2%p 이상의 분산이 있을 가능성.
*   **ResNet50 학습 중단 시점이 이름**: val_loss가 monotonic 감소 중인 5 epoch에서 종료.
*   CIFAR-10 표준 augmentation 미적용: RandomCrop(32, padding=4)는 CIFAR-10 baseline에서 사실상 표준이지만 누락.



---


#5. 앞으로의 학습 방향 (ver1 방향성): 전이 학습의 한계 돌파 및 아키텍처 맞춤화

*   현재 ver 0의 비교 실험은 이를 테면 '작고 얕은 맞춤형 모델(Custom CNN, 32x32)'과 '거대하지만 억지로 해상도를 늘린 모델(ResNet50 Frozen, 224x224)' 간의 대결이었다.

*   억지로 이미지를 확대하면서 발생하는 화질 왜곡(Blur)과 피처 손실에 대한 의문을 해결하기 위해, 향후 학습은 다음 두 가지 트랙을 동시에 진행하여 딥러닝의 본질을 깊이 있게 탐구할 것이다.

<br>

📌 Track A: 모델 스스로의 적응력 극대화 (Advanced Fine-tuning)
*   목적: 32x32 ➡️ 224x224 리사이즈로 인한 이미지 왜곡을 모델 스스로 교정하도록 유도.

*  구현 계획: ResNet50의 몸통(Backbone) 동결을 풀고 네트워크 전체를 학습. 단, 기존의 지식(ImageNet)이 망가지지 않도록 아주 작은 학습률(Low Learning Rate)을 부여.

*  기대 효과: ImageNet의 선명한 화질에 맞춰져 있던 필터들이, 흐릿하고 늘어난 CIFAR 데이터의 특징에 맞춰 미세하게 스스로를 교정하는 '진정한 전이 학습'을 실현할 수 있을 것.

<br>

📌 Track B: 데이터에 모델을 맞추다 (CIFAR-tailored Small-input ResNet)
*  목적: 억지로 해상도를 키우는 대신, 32x32 원본 해상도를 잃지 않고 그대로 받아들일 수 있는 깊은 모델을 직접 설계.

*  구현 계획: ResNet의 초기 진입부(Stem)를 CIFAR-10에 맞게 개조. 픽셀을 너무 많이 압축해 버리는 7x7 컨볼루션 필터와 MaxPool 층을 3x3 필터로 교체하여, 작은 이미지의 공간 정보(Spatial Information)가 깊은 층까지 살아남도록 아키텍처를 직접 수정. (예: ResNet-20 구조 활용)

*  기대 효과: 데이터의 크기(Input Size)와 모델의 수용력(Capacity)이 어떻게 상호작용하는지, 아키텍처 레벨에서 완벽하게 이해.